In [ ]:
from google.colab import drive
import matplotlib.font_manager as fm
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import geopandas as gpd

# 1. Shapefile 지도 데이터 불러오기
shp_path = '/content/drive/MyDrive/교통문제공모전/bnd_dong_00_2021_4Q.shp'
gdf = gpd.read_file(shp_path, encoding='euc-kr')

# 2. 읍면동 코드 CSV 데이터 불러오기
commute_path = '/content/drive/MyDrive/교통문제공모전/town_name_code.CSV'
df = pd.read_csv(commute_path, encoding='utf-8')

In [ ]:
# 전체 데이터의 앞 10줄 확인하여 어떤 열(Column)에 코드가 있는지 파악
display(gdf.head(10))

,BASE_DATE,ADM_CD,ADM_NM,geometry
0,20211231,1101053,사직동,"POLYGON ((953553.932 1953335.741, 953555.211 1..."
1,20211231,1101054,삼청동,"POLYGON ((954025.242 1953916.389, 954026.972 1..."
2,20211231,1101055,부암동,"POLYGON ((952490.38 1956548.821, 952497.594 19..."
3,20211231,1101056,평창동,"POLYGON ((953683.828 1959209.871, 953665.283 1..."
4,20211231,1101057,무악동,"POLYGON ((952298.144 1953539.606, 952324.838 1..."
5,20211231,1101058,교남동,"POLYGON ((952572.048 1953258.829, 952573.174 1..."
6,20211231,1101060,가회동,"POLYGON ((954894.795 1954614.58, 954888.29 195..."
7,20211231,1101061,종로1·2·3·4가동,"POLYGON ((954918.389 1954371.538, 954926.411 1..."
8,20211231,1101063,종로5·6가동,"POLYGON ((956606.81 1953149.973, 956606.726 19..."
9,20211231,1101064,이화동,"POLYGON ((956365.89 1954112.187, 956371.71 195..."


생성형 AI(Claude)를 이용해 shp파일과 csv파일 간 행정구역 코드 비교 -> 결과물 'code_compare_result.csv' 생성

In [ ]:
compare_path = '/content/drive/MyDrive/교통문제공모전/code_compare_result.csv'
cdf = pd.read_csv(compare_path, encoding='utf-8')
cdf.head(10)

,읍면동코드,행정구역명,CSV존재여부,SHP존재여부
0,1101053,서울특별시 종로구 사직동,True,True
1,1101054,서울특별시 종로구 삼청동,True,True
2,1101055,서울특별시 종로구 부암동,True,True
3,1101056,서울특별시 종로구 평창동,True,True
4,1101057,서울특별시 종로구 무악동,True,True
5,1101058,서울특별시 종로구 교남동,True,True
6,1101060,서울특별시 종로구 가회동,True,True
7,1101061,서울특별시 종로구 종로1·2·3·4가동,True,True
8,1101063,서울특별시 종로구 종로5·6가동,True,True
9,1101064,서울특별시 종로구 이화동,True,True


In [ ]:
# 수도권 데이터(읍면동코드가 11(서울), 23(인천), 31(경기)로 시작)만 필터링
cdf_metropolitan = cdf[cdf['읍면동코드'].astype(str).str.startswith(('11', '23', '31'))]

# csv와 shp 둘 중 하나라도 없는 지역(False가 있는 지역) 필터링
cdf_f1 = cdf_metropolitan[(cdf_metropolitan['CSV존재여부'] == False) | (cdf_metropolitan['SHP존재여부'] == False)]
cdf_filtered = cdf_f1[(cdf_f1['CSV존재여부'] != False) | (cdf_f1['SHP존재여부'] != False)]
display(cdf_filtered.head())
cdf_filtered.info()

,읍면동코드,행정구역명,CSV존재여부,SHP존재여부
270,1117068,서울특별시 구로구 오류2동,True,False
275,1117073,NaN,False,True
276,1117074,NaN,False,True
408,1125051,서울특별시 강동구 강일동,True,False
409,1125052,서울특별시 강동구 상일동,True,False


<class 'pandas.core.frame.DataFrame'>
Index: 197 entries, 270 to 1813
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   읍면동코드    197 non-null    int64 
 1   행정구역명    99 non-null     object
 2   CSV존재여부  197 non-null    bool  
 3   SHP존재여부  197 non-null    bool  
dtypes: bool(2), int64(1), object(1)
memory usage: 5.0+ KB


In [ ]:
# gdf의 ADM_CD와 ADM_NM 컬럼만 선택하여 병합 준비
gdf_admin_names = gdf[['ADM_CD', 'ADM_NM']].copy()
gdf_admin_names.rename(columns={'ADM_CD': '읍면동코드'}, inplace=True)

# '읍면동코드'를 문자열로 통일하여 병합의 일관성 확보
cdf_filtered['읍면동코드'] = cdf_filtered['읍면동코드'].astype(str)
gdf_admin_names['읍면동코드'] = gdf_admin_names['읍면동코드'].astype(str)

# cdf_filtered에 gdf의 행정구역명을 병합
# 만약 cdf_filtered에도 ADM_NM과 동일한 이름의 컬럼이 있다면 자동으로 _x, _y 접미사가 붙습니다.
cdf_filtered = pd.merge(cdf_filtered,gdf_admin_names,on='읍면동코드',how='left')

# '행정구역명'이 NaN인 경우 'ADM_NM' 값으로 채우기 (gdf에서 온 컬럼)
cdf_filtered['행정구역명'] = cdf_filtered['행정구역명'].fillna(cdf_filtered['ADM_NM'])

In [ ]:
# 임시로 추가된 'ADM_NM' 컬럼 삭제
cdf_filtered.drop(columns=['ADM_NM'], inplace=True)

display(cdf_filtered.head())
cdf_filtered.info()

,읍면동코드,행정구역명,CSV존재여부,SHP존재여부
0,1117068,서울특별시 구로구 오류2동,True,False
1,1117073,오류2동,False,True
2,1117074,항동,False,True
3,1125051,서울특별시 강동구 강일동,True,False
4,1125052,서울특별시 강동구 상일동,True,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   읍면동코드    197 non-null    object
 1   행정구역명    197 non-null    object
 2   CSV존재여부  197 non-null    bool  
 3   SHP존재여부  197 non-null    bool  
dtypes: bool(2), object(2)
memory usage: 3.6+ KB


In [ ]:
#csv로 저장
cdf_path = '/content/drive/MyDrive/교통문제공모전/rematching_town_code.csv'
cdf_filtered.to_csv(cdf_path, index=False, encoding='utf-8-sig')
print(f"데이터 저장 완료: {cdf_path}")

데이터 저장 완료: /content/drive/MyDrive/교통문제공모전/rematching_town_code.csv


**SHP파일 행정동 코드 수정**
- 코드 재매칭용 csv파일 (fin_rematching_town_code.csv)은 수작업으로 비교해가며 잘못된 부분 코드 재매칭
- 재매칭 적용

In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Shapefile 지도 데이터 불러오기
shp_path = '/content/drive/MyDrive/교통문제공모전/bnd_dong_00_2021_4Q.shp'
gdf = gpd.read_file(shp_path, encoding='euc-kr')

# 2. 코드 재매칭용 CSV 데이터 불러오기
match_path = '/content/drive/MyDrive/교통문제공모전/fin_rematching_town_code.csv'
mdf = pd.read_csv(match_path, encoding='utf-8')

mdf['기존코드'] = mdf['기존코드'].astype(str)
mdf['바꿀코드'] = mdf['바꿀코드'].astype(str)

# '기존코드'를 key로, '바꿀코드'를 value로 가지는 딕셔너리 생성
code_map = dict(zip(mdf['기존코드'], mdf['바꿀코드']))

# 4. SHP 파일의 코드 변경
# ('ADM_CD' 부분을 실제 SHP 파일 내의 행정동 코드 컬럼명으로 변경)
target_col = 'ADM_CD'

gdf[target_col] = gdf[target_col].astype(str) # 원본 컬럼도 문자열로 통일
gdf[target_col] = gdf[target_col].replace(code_map)

# 5. 수정된 Shapefile 저장
output_shp_path = "updated_shapefile.shp"
gdf.to_file(output_shp_path, encoding='euc-kr')

print("행정동 코드 수정 및 저장이 완료되었습니다!")

행정동 코드 수정 및 저장이 완료되었습니다!
